# N-BaIoT Preprocessing — Step 4: Train/Test Split
**Device: Danmini Doorbell**

This notebook uses the Step 3 labelled CSV and creates separate training and testing datasets.

### What this step does
1. Count the samples in each class.
2. Randomly select the training rows separately within each class.
3. Write the training and testing CSVs in chunks to avoid loading the entire dataset into RAM.
4. Verify that each class appears in both splits and that the requested per-class split sizes are exact.

**Split:** 80% training / 20% testing (stratified by `label_id`).

This is a within-device split for Danmini Doorbell. It is not a cross-device/generalization test. Feature scaling and model training happen later, and any scaler must be fitted on training data only.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

# ===== CONFIGURATION =====
DEVICE = "Danmini_Doorbell"
TRAIN_RATIO = 0.80
RANDOM_SEED = 42
CHUNK_SIZE = 250_000

if not 0 < TRAIN_RATIO < 1:
    raise ValueError("TRAIN_RATIO must be between 0 and 1.")

# Locate the preprocessed folder.
cwd = Path.cwd()
BASE_DIR = None
for root in [cwd, *cwd.parents]:
    if (root / "preprocessed").is_dir():
        BASE_DIR = root
        break

if BASE_DIR is None:
    matches = [p for p in cwd.rglob("preprocessed") if p.is_dir()]
    if matches:
        BASE_DIR = matches[0].parent

if BASE_DIR is None:
    raise FileNotFoundError("Could not find 'preprocessed'. Run Steps 1–3 first.")

PRE_DIR = BASE_DIR / "preprocessed"
INPUT_PATH = PRE_DIR / f"{DEVICE}_step3_labelled.csv"
MAPPING_PATH = PRE_DIR / f"{DEVICE}_label_mapping.json"
TRAIN_PATH = PRE_DIR / f"{DEVICE}_train.csv"
TEST_PATH = PRE_DIR / f"{DEVICE}_test.csv"
REPORT_PATH = PRE_DIR / f"{DEVICE}_step4_split_report.json"

if not INPUT_PATH.is_file():
    raise FileNotFoundError(f"Step 3 file not found: {INPUT_PATH}. Run Step 3 first.")
if not MAPPING_PATH.is_file():
    raise FileNotFoundError(f"Label mapping not found: {MAPPING_PATH}. Run Step 3 first.")

with open(MAPPING_PATH, "r", encoding="utf-8") as f:
    label_mapping = json.load(f)

label_to_id = label_mapping["label_to_id"]
print("Input :", INPUT_PATH)
print("Train :", TRAIN_PATH)
print("Test  :", TEST_PATH)
print(f"Split : {TRAIN_RATIO:.0%} train / {1-TRAIN_RATIO:.0%} test")
print("Seed  :", RANDOM_SEED)


---
## 4.1 Count samples in each class

We make a first pass through the CSV, reading only the two label columns. This gives the exact number of rows in each class without loading all 115 features into memory.


In [ ]:
class_counts = {}
total_rows = 0

for chunk in pd.read_csv(
    INPUT_PATH,
    usecols=["label", "label_id"],
    chunksize=CHUNK_SIZE
):
    if chunk[["label", "label_id"]].isnull().any().any():
        raise ValueError("Missing label or label_id found in Step 3 file.")

    for label_id, count in chunk["label_id"].value_counts().items():
        key = int(label_id)
        class_counts[key] = class_counts.get(key, 0) + int(count)
    total_rows += len(chunk)

if total_rows == 0:
    raise ValueError("The Step 3 dataset is empty.")

id_to_label = {int(v): k for k, v in label_to_id.items()}
unknown_ids = set(class_counts) - set(id_to_label)
if unknown_ids:
    raise ValueError(f"label_id values missing from mapping: {sorted(unknown_ids)}")

print(f"Total samples: {total_rows:,}")
print("\nClass counts:")
for label_id in sorted(class_counts):
    label = id_to_label[label_id]
    print(f"{label:22s} (ID {label_id:2d}): {class_counts[label_id]:>12,}")


---
## 4.2 Select an exact stratified split

For each class, we randomly choose exactly 80% of its rows for training. The remaining rows go to testing. This preserves each class's representation and gives exact per-class split counts.

We store only a small Boolean selection array per class, not the full feature dataset.


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
train_flags = {}
expected_train_counts = {}
expected_test_counts = {}

for label_id, count in class_counts.items():
    if count < 2:
        raise ValueError(
            f"Class {id_to_label[label_id]!r} has only {count} row(s); "
            "it needs at least 2 rows for a train/test split."
        )

    n_train = int(round(count * TRAIN_RATIO))
    n_train = min(max(n_train, 1), count - 1)

    flags = np.zeros(count, dtype=bool)
    selected = rng.choice(count, size=n_train, replace=False)
    flags[selected] = True

    train_flags[label_id] = flags
    expected_train_counts[label_id] = n_train
    expected_test_counts[label_id] = count - n_train

print("Exact per-class split sizes:")
for label_id in sorted(class_counts):
    label = id_to_label[label_id]
    print(
        f"{label:22s}: train {expected_train_counts[label_id]:>10,} | "
        f"test {expected_test_counts[label_id]:>10,}"
    )


---
## 4.3 Write train and test CSVs in chunks

We read the full feature data in chunks and route each row according to the precomputed class-specific selection. The original `label` and `label_id` columns are retained in both files.


In [ ]:
# Remove previous outputs so rerunning the cell does not append duplicate rows.
for path in [TRAIN_PATH, TEST_PATH]:
    if path.exists():
        path.unlink()

class_positions = {label_id: 0 for label_id in class_counts}
train_rows_written = 0
test_rows_written = 0
first_train = True
first_test = True

for chunk_number, chunk in enumerate(
    pd.read_csv(INPUT_PATH, chunksize=CHUNK_SIZE, low_memory=False),
    start=1
):
    if not {"label", "label_id"}.issubset(chunk.columns):
        raise ValueError("Step 3 input must contain 'label' and 'label_id'.")

    train_mask = np.zeros(len(chunk), dtype=bool)
    ids = chunk["label_id"].to_numpy()

    for label_id in np.unique(ids):
        label_id = int(label_id)
        if label_id not in train_flags:
            raise ValueError(f"Unexpected label_id in input: {label_id}")

        row_positions = np.flatnonzero(ids == label_id)
        start = class_positions[label_id]
        end = start + len(row_positions)

        if end > len(train_flags[label_id]):
            raise ValueError(f"More rows found for label_id {label_id} than counted in pass 1.")

        train_mask[row_positions] = train_flags[label_id][start:end]
        class_positions[label_id] = end

    train_chunk = chunk.loc[train_mask]
    test_chunk = chunk.loc[~train_mask]

    if len(train_chunk):
        train_chunk.to_csv(
            TRAIN_PATH,
            mode="w" if first_train else "a",
            header=first_train,
            index=False
        )
        first_train = False
        train_rows_written += len(train_chunk)

    if len(test_chunk):
        test_chunk.to_csv(
            TEST_PATH,
            mode="w" if first_test else "a",
            header=first_test,
            index=False
        )
        first_test = False
        test_rows_written += len(test_chunk)

    print(
        f"Chunk {chunk_number:>3}: "
        f"train {len(train_chunk):>9,} | test {len(test_chunk):>9,}"
    )

if any(class_positions[label_id] != class_counts[label_id] for label_id in class_counts):
    raise RuntimeError("Second pass row counts did not match the first pass.")

print("\n✓ Split files written.")
print(f"Training rows: {train_rows_written:,}")
print(f"Testing rows : {test_rows_written:,}")


---
## 4.4 Verify the split

We count labels in each output file and compare them with the exact expected per-class counts. The overall row totals must also match the input.


In [ ]:
def count_split_labels(path):
    counts = {}
    rows = 0
    for chunk in pd.read_csv(path, usecols=["label_id"], chunksize=CHUNK_SIZE):
        rows += len(chunk)
        for label_id, count in chunk["label_id"].value_counts().items():
            key = int(label_id)
            counts[key] = counts.get(key, 0) + int(count)
    return rows, counts

actual_train_rows, actual_train_counts = count_split_labels(TRAIN_PATH)
actual_test_rows, actual_test_counts = count_split_labels(TEST_PATH)

assert actual_train_rows == train_rows_written
assert actual_test_rows == test_rows_written
assert actual_train_rows + actual_test_rows == total_rows

print("=== TRAIN / TEST DISTRIBUTION ===")
print(f"{'Class':22s} {'Train':>12s} {'Test':>12s} {'Total':>12s}")
for label_id in sorted(class_counts):
    label = id_to_label[label_id]
    tr = actual_train_counts.get(label_id, 0)
    te = actual_test_counts.get(label_id, 0)
    expected_tr = expected_train_counts[label_id]
    expected_te = expected_test_counts[label_id]
    assert tr == expected_tr, f"Train count mismatch for {label}"
    assert te == expected_te, f"Test count mismatch for {label}"
    print(f"{label:22s} {tr:12,} {te:12,} {tr+te:12,}")

print("\n✓ Exact per-class counts verified.")
print("✓ Every class is represented in both splits.")
print("✓ Train + test rows equal the Step 3 input rows.")


---
## 4.5 Save the split report

The report records the random seed, split ratio, total row counts, and per-class counts so the experiment is reproducible.


In [ ]:
report = {
    "device": DEVICE,
    "input_file": str(INPUT_PATH),
    "train_file": str(TRAIN_PATH),
    "test_file": str(TEST_PATH),
    "train_ratio": TRAIN_RATIO,
    "test_ratio": 1 - TRAIN_RATIO,
    "random_seed": RANDOM_SEED,
    "total_rows": total_rows,
    "train_rows": actual_train_rows,
    "test_rows": actual_test_rows,
    "class_distribution": {
        id_to_label[label_id]: {
            "label_id": label_id,
            "total": class_counts[label_id],
            "train": actual_train_counts.get(label_id, 0),
            "test": actual_test_counts.get(label_id, 0)
        }
        for label_id in sorted(class_counts)
    },
    "note": (
        "Stratified within-device split. This is not a cross-device test. "
        "Fit any preprocessing scalers on training data only."
    )
}

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("✓ Step 4 complete!")
print("Training dataset :", TRAIN_PATH)
print("Testing dataset  :", TEST_PATH)
print("Split report     :", REPORT_PATH)
print(f"Train file size  : {TRAIN_PATH.stat().st_size / (1024**2):.1f} MB")
print(f"Test file size   : {TEST_PATH.stat().st_size / (1024**2):.1f} MB")
print("\n🚀 Ready for the next stage: model training and evaluation.")
